# Custom Mixed-Precision Quantization → `.pte` (ExecuTorch / Android)

| Property | Value |
|----------|-------|
| **Model** | Any Gemma-compatible model (pruned, distilled, or base) |
| **Source** | HuggingFace Hub **or** local directory |
| **Backend** | ExecuTorch XNNPACK (ARM CPU) |
| **Quantization** | Native INT8 weights (1 byte/param) + FP16 scales & norms |
| **Output** | `~0.9 GB .pte` for a 1B-parameter model |
| **Platform** | Google Colab (TPU v5e recommended for ≥48 GB system RAM) |

## Workflow

```
Friend trains / prunes / distills a Gemma model
        ↓
Uploads to HuggingFace  —or—  gives you a local folder
        ↓
You set MODEL_PATH (Cell 5)  →  run this notebook
        ↓
Get a .pte file  →  deploy in Android app
```

## How to Run

1. **Set `MODEL_PATH`** in Cell 5 to your model (HF repo ID or local path).
2. **Set `HF_TOKEN`** in Cell 4 via Colab Secrets (recommended) or paste directly.
3. Run **Cell 2** (Install). Wait for it to finish.
4. **Runtime → Restart session.**
5. Run **Cells 3 → 4 → 5 → 6** sequentially.

> **Note:** The `FutureWarning: isinstance(treespec, LeafSpec)` messages during lowering are harmless and can be ignored.


In [1]:
# ── CELL 2: INSTALL DEPENDENCIES ──────────────────────────────────────────────
# Run once, then Runtime → Restart session.
#
# Colab ships its own torch build. We need a specific nightly CPU build that
# is ABI-compatible with executorch.  `pip uninstall` alone leaves remnant
# files (the "~orch" directories), so we physically delete them first.
# ──────────────────────────────────────────────────────────────────────────────

import subprocess, sys, os, shutil, glob

# 1. Locate site-packages
sp = subprocess.run(
    [sys.executable, "-c", "import site; print(site.getsitepackages()[0])"],
    capture_output=True, text=True
).stdout.strip()
print(f"site-packages: {sp}")

# 2. Uninstall (clears metadata)
!pip uninstall -y torch torchvision torchaudio torch-xla torchao executorch 2>/dev/null || true

# 3. Physical deletion of leftover directories
patterns = [
    "torch", "torch-*", "torch_*",
    "torchvision*", "torchaudio*", "torch_xla*",
    "torchao*", "executorch*", "~orch*",
]
removed = []
for pat in patterns:
    for path in glob.glob(os.path.join(sp, pat)):
        shutil.rmtree(path, ignore_errors=True)
        removed.append(os.path.basename(path))
print(f"Removed {len(removed)} leftover dir(s).") if removed else print("Clean.")

!pip cache purge 2>/dev/null || true

# 4. Verify torch is truly gone
r = subprocess.run([sys.executable, "-c", "import torch"], capture_output=True, text=True)
if r.returncode == 0:
    !rm -rf {sp}/torch {sp}/torch-* {sp}/~orch*
    print("WARNING: had to force-remove extra torch remnants.")
else:
    print("torch fully removed.")

# 5. Install torch (CPU nightly) → executorch + torchao
print("\nInstalling torch (CPU nightly)...")
!pip install --no-cache-dir --pre torch \
    --index-url https://download.pytorch.org/whl/nightly/cpu

print("\nInstalling executorch + torchao...")
!pip install --no-cache-dir executorch torchao \
    --extra-index-url https://download.pytorch.org/whl/nightly/cpu

# 6. HuggingFace stack
print("\nInstalling HuggingFace stack...")
!pip install --quiet -U "transformers>=4.50" "tokenizers>=0.21" \
    accelerate sentencepiece huggingface_hub

# 7. Diagnostic
print("\n" + "=" * 60)
diag = subprocess.run([sys.executable, "-c", """
import torch, transformers
print(f"  torch          : {torch.__version__}")
try:
    import torchao; print(f"  torchao        : {torchao.__version__}")
except: print("  torchao        : NOT INSTALLED")
try:
    import executorch; print(f"  executorch     : {getattr(executorch, '__version__', 'installed')}")
except: print("  executorch     : NOT INSTALLED")
print(f"  transformers   : {transformers.__version__}")

checks = [
    ("XnnpackPartitioner",          "executorch.backends.xnnpack.partition.xnnpack_partitioner"),
    ("to_edge_transform_and_lower", "executorch.exir"),
    ("EdgeCompileConfig",           "executorch.exir"),
]
for sym, mod in checks:
    try:
        m = __import__(mod, fromlist=[sym])
        assert getattr(m, sym, None) is not None
        print(f"  {sym:35s} ✓")
    except Exception as e:
        print(f"  {sym:35s} ✗ ({e})")
"""], capture_output=True, text=True)
print(diag.stdout)
if diag.returncode != 0:
    print("STDERR:", diag.stderr[-500:])
print("=" * 60)
print("\n✅ Done.  Runtime → Restart session → run cells 3 onwards.")


site-packages: /usr/local/lib/python3.12/dist-packages
Found existing installation: torch 2.9.0+cpu
Uninstalling torch-2.9.0+cpu:
  Successfully uninstalled torch-2.9.0+cpu
Found existing installation: torchvision 0.24.0+cpu
Uninstalling torchvision-0.24.0+cpu:
  Successfully uninstalled torchvision-0.24.0+cpu
Found existing installation: torchaudio 2.9.0+cpu
Uninstalling torchaudio-2.9.0+cpu:
  Successfully uninstalled torchaudio-2.9.0+cpu
Found existing installation: torch-xla 2.9.0
Uninstalling torch-xla-2.9.0:
  Successfully uninstalled torch-xla-2.9.0
Clean.
Files removed: 0
torch fully removed.

Installing torch (CPU nightly)...
Looking in indexes: https://download.pytorch.org/whl/nightly/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.0/193.0 MB 51.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0


Installing HuggingFace stack...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 137.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 45.7 MB/s eta 0:00:00

  torch          : 2.12.0.dev20260301+cpu
  torchao        : 0.15.0
  executorch     : installed
  transformers   : 5.2.0
  XnnpackPartitioner                  ✓
  to_edge_transform_and_lower         ✓
  EdgeCompileConfig                   ✓


✅ Done.  Runtime → Restart session → run cells 3 onwards.


In [1]:
# ── CELL 3: IMPORT RESOLVER ───────────────────────────────────────────────────
# Probes all known module paths for every symbol the pipeline needs.
# Adapts automatically to whatever torch / torchao / executorch versions
# are installed, so the notebook never hits a hard-coded ImportError.
# ──────────────────────────────────────────────────────────────────────────────

import importlib, torch
print(f"torch {torch.__version__}\n")

def _find(symbol: str, candidates: list[str]):
    """Return the first matching object from candidate module paths, or None."""
    for mod_path in candidates:
        try:
            mod = importlib.import_module(mod_path)
            obj = getattr(mod, symbol, None)
            if obj is not None:
                print(f"  ✅ {symbol:40s} ← {mod_path}")
                return obj
        except Exception:
            pass
    print(f"  ❌ {symbol:40s}   (not found)")
    return None

# ── PT2E quantization (optional — may not be available in nightly) ───────────
prepare_pt2e = _find("prepare_pt2e", [
    "torch.ao.quantization.quantize_pt2e",
    "torchao.quantization.pt2e.quantize_pt2e",
    "torch.ao.quantization._pt2e.quantize_pt2e",
    "torch.ao.quantization.pt2e",
])
convert_pt2e = _find("convert_pt2e", [
    "torch.ao.quantization.quantize_pt2e",
    "torchao.quantization.pt2e.quantize_pt2e",
    "torch.ao.quantization._pt2e.quantize_pt2e",
    "torch.ao.quantization.pt2e",
])
XNNPACKQuantizer = _find("XNNPACKQuantizer", [
    "executorch.backends.xnnpack.quantization.xnnpack_quantizer",
    "torchao.quantization.pt2e.xnnpack_quantizer",
    "torchao.quantization.xnnpack_quantizer",
    "torchao._executorch.xnnpack_quantizer",
    "torch.ao.quantization.quantizer.xnnpack_quantizer",
])
get_sym_config = _find("get_symmetric_quantization_config", [
    "executorch.backends.xnnpack.quantization.xnnpack_quantizer",
    "torchao.quantization.pt2e.xnnpack_quantizer",
    "torchao.quantization.xnnpack_quantizer",
    "torchao._executorch.xnnpack_quantizer",
    "torch.ao.quantization.quantizer.xnnpack_quantizer",
])

# ── ExecuTorch lowering (required) ───────────────────────────────────────────
XnnpackPartitioner = _find("XnnpackPartitioner", [
    "executorch.backends.xnnpack.partition.xnnpack_partitioner",
])
to_edge_transform_and_lower = _find("to_edge_transform_and_lower", [
    "executorch.exir",
])
EdgeCompileConfig = _find("EdgeCompileConfig", [
    "executorch.exir",
])

assert XnnpackPartitioner,        "XnnpackPartitioner not found"
assert to_edge_transform_and_lower, "to_edge_transform_and_lower not found"
assert EdgeCompileConfig,         "EdgeCompileConfig not found"

# ── Decide quantization path ─────────────────────────────────────────────────
USE_PT2E = all([prepare_pt2e, convert_pt2e, XNNPACKQuantizer, get_sym_config])
USE_TORCHAO_QUANTIZE = False

if not USE_PT2E:
    try:
        from torchao.quantization import quantize_ as _tq, int8_weight_only as _i8
        USE_TORCHAO_QUANTIZE = True
    except ImportError:
        pass

path_label = (
    "PT2E + XNNPACKQuantizer"        if USE_PT2E else
    "Native INT8 module replacement" if USE_TORCHAO_QUANTIZE else
    "None (FP32 export)"
)
print(f"\n  Quantization path: {path_label}")
print("✅ Import resolution complete.")


torch 2.12.0.dev20260301+cpu



  ❌ prepare_pt2e                               (not found)
  ❌ convert_pt2e                               (not found)
  ❌ XNNPACKQuantizer                           (not found)
  ❌ get_symmetric_quantization_config          (not found)
  ✅ XnnpackPartitioner                       ← executorch.backends.xnnpack.partition.xnnpack_partitioner
  ✅ to_edge_transform_and_lower              ← executorch.exir
  ✅ EdgeCompileConfig                        ← executorch.exir

  Quantization path: Native INT8 module replacement
✅ Import resolution complete.


In [3]:
# ── CELL 4: HUGGINGFACE AUTHENTICATION ────────────────────────────────────────
# Option A (recommended): Store your token in Colab Secrets as "HF_TOKEN".
# Option B: Paste it directly below. Never commit tokens to version control.
# ──────────────────────────────────────────────────────────────────────────────

from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

HF_TOKEN = ""

# Try Colab Secrets first
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("✅ Token loaded from Colab Secrets.")
except Exception:
    pass

# Fallback: manual token (paste yours here if not using Colab Secrets)
if not HF_TOKEN:
    HF_TOKEN = ""  # ← paste your HuggingFace token here
    if HF_TOKEN:
        print("✅ Using manually provided token.")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is empty.\n"
        "  Option A: Add it to Colab Secrets (Key: HF_TOKEN).\n"
        "  Option B: Paste it in the HF_TOKEN = \"\" line above."
    )

login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to HuggingFace.")


✅ Token loaded from Colab Secrets.
Logged in to HuggingFace.


In [4]:
# ── CELL 5: LOAD MODEL + INT8 QUANTIZATION ────────────────────────────────────
#
# ★  Set MODEL_PATH to your model  ★
#
#   MODEL_PATH = "google/gemma-3-1b-it"                    # HuggingFace Hub
#   MODEL_PATH = "your-org/gemma-pruned-distilled"          # Custom HF repo
#   MODEL_PATH = "/content/drive/MyDrive/my_gemma_model"    # Google Drive
#   MODEL_PATH = "/content/my_model"                        # Uploaded folder
# ──────────────────────────────────────────────────────────────────────────────

MODEL_PATH   = "google/gemma-3-1b-it"
EXPORT_DTYPE = torch.float16   # fp16 for scales, norms, biases

import os
import torch.nn.functional as F

is_local = os.path.isdir(MODEL_PATH)
print(f"Model : {MODEL_PATH}  ({'local' if is_local else 'HuggingFace Hub'})")
print(f"Dtype : {EXPORT_DTYPE}")

# ── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH, token=HF_TOKEN if not is_local else None,
)

# ── Model ────────────────────────────────────────────────────────────────────
print("\nLoading model weights...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    token=HF_TOKEN if not is_local else None,
    device_map="cpu",
    torch_dtype=EXPORT_DTYPE,
    use_cache=False,
    attn_implementation="eager",
)
base_model.eval()

n_params = sum(p.numel() for p in base_model.parameters())
print(f"  Parameters   : {n_params / 1e6:.1f} M")
print(f"  Linear layers: {sum(1 for m in base_model.modules() if isinstance(m, torch.nn.Linear))}")

# ═══════════════════════════════════════════════════════════════════════════════
# NATIVE INT8 MODULE REPLACEMENT
#
# We replace every nn.Linear and nn.Embedding with custom modules that store
# weights as torch.int8 (1 byte per element).  A per-channel FP16 scale is
# kept alongside.  Dequantization happens inside forward(), becomes part of
# the traced graph, and the .pte serialises int8 buffers at 1 byte each.
#
# Weight tying (e.g. lm_head ↔ embed_tokens) is detected and preserved so
# that torch.export stores the shared buffer only once.
# ═══════════════════════════════════════════════════════════════════════════════

class Int8Linear(torch.nn.Module):
    """nn.Linear replacement storing weights as int8."""
    def __init__(self, weight_int8, scale, bias):
        super().__init__()
        self.register_buffer("weight_q",     weight_int8)   # [out, in], int8
        self.register_buffer("weight_scale", scale)          # [out, 1],  fp16
        self.register_buffer("linear_bias",  bias)           # [out] or None

    def forward(self, x):
        w = self.weight_q.to(self.weight_scale.dtype) * self.weight_scale
        return F.linear(x, w, self.linear_bias)


class Int8Embedding(torch.nn.Module):
    """nn.Embedding replacement storing weights as int8."""
    def __init__(self, weight_int8, scale, padding_idx):
        super().__init__()
        self.register_buffer("weight_q",     weight_int8)   # [vocab, dim], int8
        self.register_buffer("weight_scale", scale)          # [vocab, 1],   fp16
        self.padding_idx = padding_idx

    def forward(self, input_ids):
        w = self.weight_q.to(self.weight_scale.dtype) * self.weight_scale
        return F.embedding(input_ids, w, self.padding_idx)


def _quantize_weight(w_fp):
    """Symmetric per-channel quantization: fp → int8 + scale."""
    w = w_fp.float()
    scale = (w.abs().amax(dim=1, keepdim=True) / 127.0).clamp(min=1e-8)
    wq = (w / scale).round().clamp(-128, 127).to(torch.int8)
    return wq, scale


def replace_with_int8(model, dtype):
    """Swap all Linear & Embedding layers for native-int8 versions."""
    replacements = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            wq, sc = _quantize_weight(module.weight.data)
            bias = module.bias.data.to(dtype) if module.bias is not None else None
            replacements.append((name, Int8Linear(wq, sc.to(dtype), bias), "linear"))
        elif isinstance(module, torch.nn.Embedding):
            wq, sc = _quantize_weight(module.weight.data)
            replacements.append((name, Int8Embedding(wq, sc.to(dtype), module.padding_idx), "embed"))

    for name, new_mod, _ in replacements:
        parts = name.split(".")
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], new_mod)

    # Re-tie shared weights (e.g. lm_head ↔ embed_tokens)
    embeds  = [(n, m) for n, m, t in replacements if t == "embed"]
    linears = [(n, m) for n, m, t in replacements if t == "linear"]
    tied = 0
    for en, em in embeds:
        for ln, lm in linears:
            if lm.weight_q.shape == em.weight_q.shape and torch.equal(lm.weight_q, em.weight_q):
                lm.weight_q = em.weight_q
                lm.weight_scale = em.weight_scale
                tied += 1
                print(f"  Tied: {ln} ↔ {en}")

    n_lin = sum(1 for _, _, t in replacements if t == "linear")
    n_emb = sum(1 for _, _, t in replacements if t == "embed")
    return n_lin, n_emb, tied


# ── Apply INT8 replacement ───────────────────────────────────────────────────
if USE_TORCHAO_QUANTIZE or not USE_PT2E:
    n_lin, n_emb, n_tied = replace_with_int8(base_model, EXPORT_DTYPE)

    # Size estimate
    int8_bytes, scale_bytes, other_bytes, seen = 0, 0, 0, set()
    for _, mod in base_model.named_modules():
        if isinstance(mod, (Int8Linear, Int8Embedding)):
            ptr = mod.weight_q.data_ptr()
            if ptr not in seen:
                int8_bytes  += mod.weight_q.numel()
                scale_bytes += mod.weight_scale.numel() * mod.weight_scale.element_size()
                seen.add(ptr)
            if isinstance(mod, Int8Linear) and mod.linear_bias is not None:
                other_bytes += mod.linear_bias.numel() * mod.linear_bias.element_size()
    for p in base_model.parameters():
        other_bytes += p.numel() * p.element_size()

    total = int8_bytes + scale_bytes + other_bytes
    print(f"\n  Replaced {n_lin} Linear + {n_emb} Embedding → INT8")
    print(f"  Weight ties preserved: {n_tied}")
    print(f"  Estimated .pte size : ~{total / 1e9:.2f} GB")

# ── Export wrapper ───────────────────────────────────────────────────────────
class GemmaExportWrapper(torch.nn.Module):
    """Tensor-in / tensor-out wrapper for ExecuTorch export."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits

wrapped_model = GemmaExportWrapper(base_model)
print("\n✅ Model loaded, quantized (INT8), and wrapped.")


Model : google/gemma-3-1b-it  (HuggingFace Hub)
Dtype : torch.float16


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

  Parameters   : 999.9 M
  Linear layers: 183
  Tied: lm_head ↔ model.embed_tokens

  Replaced 183 Linear + 1 Embedding → INT8
  Weight ties preserved: 1
  Estimated .pte size : ~1.00 GB

✅ Model loaded, quantized (INT8), and wrapped.


In [5]:
# ── CELL 6: TRACE + LOWER + EXPORT .PTE ────────────────────────────────────────
import os, re, gc, shutil
from torch.export import export, Dim

print("Trace → Lower → Save .pte\n")

# ── Example inputs + dynamic shapes ──────────────────────────────────────────
dummy_ids  = torch.randint(0, tokenizer.vocab_size, (1, 128), dtype=torch.long)
dummy_mask = torch.ones(1, 128, dtype=torch.long)
example_args = (dummy_ids, dummy_mask)

seq = Dim("seq_len", min=1, max=512)
dyn_shapes = ({1: seq}, {1: seq})

# ── Trace ─────────────────────────────────────────────────────────────────────
print("  Tracing model graph (high RAM — several minutes)...")
exported_model = export(wrapped_model, example_args, dynamic_shapes=dyn_shapes)
print("  ✅ Traced.")

# ── PT2E quantization (only if XNNPACKQuantizer was found) ────────────────────
if USE_PT2E:
    quantizer = XNNPACKQuantizer()
    quant_config = get_sym_config(is_per_channel=True, is_dynamic=False)
    quantizer.set_global(quant_config)
    prepared = prepare_pt2e(exported_model, quantizer)
    exported_model = convert_pt2e(prepared)
    print("  ✅ PT2E quantization applied.")
else:
    print("  ✅ Weights already INT8 — skipping post-trace quantization.")

gc.collect()

# ── Lower to Edge IR + XNNPACK backend ───────────────────────────────────────
print("  Lowering to Edge IR + XNNPACK backend...")
executorch_program = to_edge_transform_and_lower(
    exported_model,
    partitioner=[XnnpackPartitioner()],
    compile_config=EdgeCompileConfig(_check_ir_validity=False),
).to_executorch()

# ── Save .pte ─────────────────────────────────────────────────────────────────
_slug = re.sub(r"[^a-zA-Z0-9_-]", "_", MODEL_PATH.rstrip("/").split("/")[-1])
_dtype_tag = "int8" if (USE_PT2E or USE_TORCHAO_QUANTIZE) else str(EXPORT_DTYPE).split(".")[-1]
output_filename = f"{_slug}_{_dtype_tag}.pte"

with open(output_filename, "wb") as f:
    f.write(executorch_program.buffer)

size_mb = os.path.getsize(output_filename) / (1024 * 1024)
print(f"\n{'='*60}")
print(f"  ✅  SUCCESS!")
print(f"  Model  : {MODEL_PATH}")
print(f"  File   : {output_filename}")
print(f"  Size   : {size_mb:.2f} MB  ({size_mb/1024:.2f} GB)")
print(f"  Storage: INT8 weights (1 byte) + {EXPORT_DTYPE} norms/scales")
print(f"{'='*60}")

# ── Save to Google Drive ─────────────────────────────────────────────────────
DRIVE_DIR = "/content/drive/MyDrive/ExecuTorch_Models"

try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    drive_path = os.path.join(DRIVE_DIR, output_filename)
    shutil.copy2(output_filename, drive_path)
    print(f"\n💾 Saved to Google Drive: {drive_path}")
except ImportError:
    print("\n⚠️  Not running on Colab — skipping Google Drive save.")
    print(f"    File is available locally: {os.path.abspath(output_filename)}")

print(f"\n📱 Android deployment:")
print(f"  1. Download {output_filename} (or grab from Drive)")
print(f"  2. Copy to your app's assets/ folder")
print(f"  3. Load:  Module module = Module.load(assetFilePath(\"{output_filename}\"));")
print(f"  4. Run :  module.forward(inputIds, attentionMask);")


Trace → Lower → Save .pte

  Tracing model graph (high RAM — several minutes)...
  ✅ Traced.
  ✅ Weights already INT8 — skipping post-trace quantization.
  Lowering to Edge IR + XNNPACK backend...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)



  ✅  SUCCESS!
  Model  : google/gemma-3-1b-it
  File   : gemma-3-1b-it_int8.pte
  Size   : 956.19 MB  (0.93 GB)
  Storage: INT8 weights (1 byte) + torch.float16 norms/scales
Mounted at /content/drive

💾 Saved to Google Drive: /content/drive/MyDrive/ExecuTorch_Models/gemma-3-1b-it_int8.pte

📱 Android deployment:
  1. Download gemma-3-1b-it_int8.pte (or grab from Drive)
  2. Copy to your app's assets/ folder
  3. Load:  Module module = Module.load(assetFilePath("gemma-3-1b-it_int8.pte"));
  4. Run :  module.forward(inputIds, attentionMask);
